In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Blocos combinacionais de somadores em MyHDL.

Este modulo declara implementacoes de:
- meio somador (half adder),
- somador completo (full adder),
- somador de 2 bits,
- somador generico por encadeamento,
- somador vetorial comportamental.
"""

from myhdl import *


@block
def halfAdder(a, b, soma, carry):
    """Meio somador de 1 bit.

    Args:
        a: Entrada de 1 bit.
        b: Entrada de 1 bit.
        soma: Saida de soma.
        carry: Saida de carry.
    """
    @always_comb
    def comb():
        soma.next = a ^ b
        carry.next = a & b
    return instances()


@block
def fullAdder(a, b, c, soma, carry):
    """Somador completo de 1 bit.

    Args:
        a: Primeira entrada de 1 bit.
        b: Segunda entrada de 1 bit.
        c: Carry de entrada.
        soma: Saida de soma.
        carry: Carry de saida.
    """
    @always_comb
    def comb():
        soma.next = a ^ b ^ c
        carry.next = (a & b) | (c & (a ^ b)) 
    return instances()


@block
def adder2bits(x, y, soma, carry):
    """Somador de 2 bits.

    Implementacao esperada com dois full adders, gerando
    uma soma de 2 bits e carry final.

    Args:
        x: Vetor de entrada de 2 bits.
        y: Vetor de entrada de 2 bits.
        soma: Vetor de saida de 2 bits.
        carry: Carry de saida.
    """
    c1 = Signal(bool(0)) 

    fa0 = fullAdder(
        a=x[0],
        b=y[0],
        c=0,
        soma=soma[0],
        carry=c1)

    fa1 = fullAdder(
        a=x[1],
        b=y[1],
        c=c1,
        soma=soma[1],
        carry=carry)
    return instances()



@block
def adder(x, y, soma, carry):

    n = len(x)

    # carries internos (n-1 apenas)
    carries = [Signal(bool(0)) for _ in range(n-1)]
    faList = [None for _ in range(n)]
    for i in range(n):

        if i == 0:
            cin = Signal(bool(0))   # carry inicial
        else:
            cin = carries[i-1]

        if i == n-1:
            cout = carry            # último vai para carry final
        else:
            cout = carries[i]

        faList[i] = fullAdder(
            a=x[i],
            b=y[i],
            c=cin,
            soma=soma[i],
            carry=cout)
    return instances()


@block
def addervb(x, y, soma, carry):
    """Somador vetorial em estilo comportamental.

    Versao combinacional que pode usar operacoes aritmeticas diretas
    sobre os vetores para gerar soma e carry.

    Args:
        x: Vetor de entrada.
        y: Vetor de entrada.
        soma: Vetor de saida.
        carry: Carry de saida.
    """
    @always_comb
    def comb():
        n = len(soma)
        result = intbv(0)[n+1:]
        result[:] = x + y
        soma.next = result[n:0]
        carry.next = result[n]
    return instances()


In [2]:
# -*- coding: utf-8 -*-
"""Exercice 1

Implemente a equacão:

q = a or !b
"""


from myhdl import *


@block
def exe1(q, a, b):
    """
    q = a or !b
    """

    @always_comb
    def comb():
        q.next = a or not b

    return instances()


@block
def exe2(q, a, b, c):
    """
    Implemente a tabela verdade a seguir:

    A B C | Q
    ------|--
    0 0 0 | 1
    0 0 1 | 0
    0 1 0 | 0
    0 1 1 | 1
    1 0 0 | 1
    1 0 1 | 0
    1 1 0 | 0
    1 1 1 | 1

    Não utilize IF!
    """

    @always_comb
    def comb():
        q.next = (not b and not c) or (b and c)

    return instances()


@block
def exe3(q, a, b, c, d, e):
    """
    Exercice 3

    Implemente o circuito lógico a seguir:

               __
    a---------\  \
              |   )-  __
    b---------/__/  -|  \
                     |   )-
    c----------------|__/  -  __
                            -|  \
                             |   )-
    d------------------------|__/  -  __
                                    -|  \
                                     |   )-----
    e--------------------------------|__/
    """

    @always_comb
    def comb():
        q.next = ( ( (a or b) and c ) and d) and e

    return instances()


@block
def exe4(led, sw):
    """
    led0 é sw[0] and (não sw[1])
    """

    @always_comb
    def comb():
        led[0].next = sw[0] and (not sw[1])

    return instances()


@block
def exe5(leds, sw):
    """
    led0 é uma copia da chave sw0
    led1 é sw0 & sw1
    led2 é o led1 invertido
    led3 é xor entre sw0 e sw1
    todos os outros leds acesos
    """

    @always_comb
    def comb():
        leds[0].next = sw[0]
        leds[1].next = sw[0] and sw[1]
        leds[2].next = not (sw[0] and sw[1])
        leds[3].next = (not sw[0] and sw[1]) or (sw[0] and not sw[1])
        for i in range(4,10):
            leds[i]=1

    return instances()


@block
def sw2hex(hex_pins, sw):
    """
    Faz a conversão de binário para display de 7 segmentos
    """

    @always_comb
    def comb():
        if sw[4:0] == 0:
            hex_pins.next = "1000000"
        elif sw[4:0] == 1:
            hex_pins.next = "1111001"
        elif sw[4:0] == 2:
            hex_pins.next = "0100100"
        elif sw[4:0] == 3:
            hex_pins.next = "0110000"
        elif sw[4:0] == 4:
            hex_pins.next = "0011001"
        elif sw[4:0] == 5:
            hex_pins.next = "0010010"
        elif sw[4:0] == 6:
            hex_pins.next = "000010"
        elif sw[4:0] == 7:
            hex_pins.next = "1111000"
        elif sw[4:0] == 8:
            hex_pins.next = "0000000"
        elif sw[4:0] == 9:
            hex_pins.next = "0010000"
        elif sw[4:0] == 10:
            hex_pins.next = "0001000"
        elif sw[4:0] == 11:
            hex_pins.next = "0000011"
        elif sw[4:0] == 12:
            hex_pins.next = "1000110"
        elif sw[4:0] == 13:
            hex_pins.next = "0100001"
        elif sw[4:0] == 14:
            hex_pins.next = "0000110"
        else:
            hex_pins.next = "0001110"

    return instances()